In [14]:
import numpy as np

class MLP:
    def __init__(self, input_size, hidden_size, output_size=1, learning_rate=0.01):
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))

        self.lr = learning_rate

    def relu(self, z):
        return np.maximum(z, 0.0)
    
    def relu_deriv(self, z):
        return (z > 0).astype(float)
    
    def sigmoid(self, z):
        z = np.maximum(-500, np.minimum(z, 500))
        return 1.0 / (1 + np.exp(-z))
    
    def compute_loss(self, Y, y_pred):
        """
        :param Y: [batch_size, 1]
        :param y_pred: [batch_size, 1]
        """
        y_pred = np.maximum(1e-7, np.minimum(y_pred, 1 - 1e-7))
        return - (Y * np.log(y_pred) + (1 - Y) * np.log(1 - y_pred))
    
    def forward(self, x):
        """
        :param x: 输入特征 [batch_size, input_size]
        """
        self.z1 = np.dot(x, self.W1) + self.b1  # [B,H]
        self.A1 = self.relu(self.z1)            # [B,H]
        self.z2 = np.dot(self.A1, self.W2) + self.b2    # [B,1]
        self.A2 = self.sigmoid(self.z2)         # [B,1]

        return self.A2
    
    def backward(self, X, Y):
        """
        :param X: [batch_size, input_size]
        :param Y: [batch_size, 1]
        """

        self.batch_size = X.shape[0]
        dz2 = self.A2 - Y
        # z = [ w1*A11 + w2*A12
        #       w1*A21 + w2*A22 ]
        # dz1 / dw1 = (A11 + A21)/batch_size = A转至后第一行点乘单位向量
        # dz2 / dw2 = (A12 + A22)/batch_size = A转至后第二行点乘单位向量
        dw2 = (1.0 / self.batch_size ) * np.dot(self.A1.T, dz2)
        db2 = (1.0 / self.batch_size) * np.sum(dz2, axis=0, keepdims=True)

        # [z1   =   [w1*A11 + w2*A12
        #  z2]       w1*A21 + w2*A22 ]
        # dL/dA11 = dL/dz1 * dz1/dA11 = (self.A2 - Y) * w1
        # A = [ A11, A12
        #       A21, A22 ]
        # dL/dA = [ (self.A2 - Y)[0] * w1, (self.A2 - Y)[0] * w2    = [ (self.A2 - Y)[0]    \dot    [w1,w2]
        #           (self.A2 - Y)[1] * w1, (self.A2 - Y)[1] * w2 ]      (self.A2 - Y)[1] ]
        # = self.A2 - Y \dot W.T
        dA1 = np.dot(dz2, self.W2.T)
        dz1 = self.relu_deriv(self.z1) * dA1

        dw1 = (1.0 / self.batch_size) * np.dot(X.T, dz1)
        db1 = (1.0 / self.batch_size) * np.sum(dz1, axis=0, keepdims=True)

        self.W1 -= self.lr * dw1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dw2
        self.b2 -= self.lr * db2

    def train(self, X, Y, epochs=10000, print_freq=1000):

        for epoch in range(epochs):
            y_pred = self.forward(X)
            self.backward(X, Y)

            if (epoch + 1) % print_freq == 0:
                total_loss = np.mean(self.compute_loss(Y, y_pred), axis=0)
                print(f"{epoch + 1} / {epochs} Epoch, Loss: {total_loss}")

        return self.W1, self.b1, self.W2, self.b2




if __name__ == "__main__":
    BATCH_SIZE = 10
    INPUT_SIZE = 8
    X = np.random.randn(BATCH_SIZE, INPUT_SIZE)
    Y = np.random.randint(0, 2, (BATCH_SIZE, 1))
    
    mlp = MLP(input_size=INPUT_SIZE, hidden_size=20, output_size=1, learning_rate=0.01)

    mlp.train(X, Y)
    # print(y_pred)


1000 / 10000 Epoch, Loss: [0.42644046]
2000 / 10000 Epoch, Loss: [0.25459053]
3000 / 10000 Epoch, Loss: [0.09986847]
4000 / 10000 Epoch, Loss: [0.03660951]
5000 / 10000 Epoch, Loss: [0.01871893]
6000 / 10000 Epoch, Loss: [0.01163384]
7000 / 10000 Epoch, Loss: [0.00805122]
8000 / 10000 Epoch, Loss: [0.0060376]
9000 / 10000 Epoch, Loss: [0.00477023]
10000 / 10000 Epoch, Loss: [0.00390927]
